In [16]:
from pathlib import Path  # Importe Path pour gérer les chemins de fichiers de façon portable
import pandas as pd  # Importe pandas pour manipuler les tableaux de données

dossier_data = Path("data") / "data"  # Définit le chemin relatif vers le dossier contenant les fichiers de données
dossier_ref = Path("ref") / "ref"  # Définit le chemin relatif vers le dossier contenant les tables de référence

stations = pd.read_csv(dossier_data / "sup_station.txt", sep=";", dtype=str, encoding="utf-8")  # Charge sup_station.txt en texte (séparateur ';', UTF-8)
supports = pd.read_csv(dossier_data / "sup_support.txt", sep=";", dtype=str, encoding="utf-8")  # Charge sup_support.txt en texte (séparateur ';', UTF-8)
nature = pd.read_csv(dossier_ref / "sup_nature.txt", sep=";", dtype=str, encoding="utf-8")  # Charge sup_nature.txt en texte (séparateur ';', UTF-8)


# 1. Évite les doublons sur les stations si une colonne identifiant de station existe
for col_sta in ["STA_NM_ANFR", "STA_ID"]:  # Teste les noms possibles d'identifiant de station
    if col_sta in stations.columns:  # Vérifie si la colonne existe dans la table stations
        stations = stations.drop_duplicates(subset=col_sta).copy()  # Supprime les doublons selon cet identifiant et évite les alertes de vue

# 2. Évite les doublons sur les supports si SUP_ID existe
if "SUP_ID" in supports.columns:  # Vérifie si la colonne SUP_ID existe dans la table supports
    supports = supports.drop_duplicates(subset="SUP_ID").copy()  # Garde un seul support par identifiant unique SUP_ID
    
# 3. Recherche une colonne commune entre stations et supports
colonnes_communes = [c for c in stations.columns if c in supports.columns]  # Liste les colonnes présentes à la fois dans stations et supports

if not colonnes_communes:  # Vérifie si aucune colonne commune n'a été trouvée
    print("\nAucune colonne commune directe entre sup_station et sup_support.")  # Affiche un message explicatif
    print("Colle les 3 listes de colonnes affichées au-dessus pour qu'on voie la bonne liaison.")  # Indique quoi vérifier pour corriger la jointure
else:  # Cas où au moins une colonne commune existe
    cle_jointure = colonnes_communes[0]  # Prend la première colonne commune trouvée comme clé de jointure
    print(f"\nJointure effectuée sur : {cle_jointure}")  # Affiche la colonne utilisée pour relier les deux tables

    df = stations.merge(supports, on=cle_jointure, how="left")  # Joint les stations aux supports en gardant toutes les lignes de stations (jointure gauche)

    # Évite aussi les doublons après jointure si nécessaire
    df = df.drop_duplicates().copy()  # Supprime d'éventuelles lignes strictement identiques après la jointure

    if "NAT_ID" in df.columns and "NAT_ID" in nature.columns:  # Vérifie si NAT_ID est présent des deux côtés pour joindre la nature
        df = df.merge(nature, on="NAT_ID", how="left")  # Ajoute les informations de nature du support via NAT_ID

    cols_nom = [c for c in nature.columns if c != "NAT_ID"]  # Récupère les colonnes de nature autres que l'identifiant NAT_ID
    col_nom = cols_nom[-1] if cols_nom else cle_jointure  # Choisit la dernière colonne de libellé disponible (ou la clé de jointure par défaut)

    total = len(df)  # Calcule le nombre total de lignes/enregistrements dans le tableau final
    top10 = df[col_nom].fillna("Non renseigné").value_counts().head(10)  # Remplace les valeurs manquantes, compte chaque type et garde les 10 plus fréquents
    print(f"\nTotal : {total}")  # Affiche l'effectif total utilisé pour les pourcentages
    print("\nTop 10 :")  # Affiche l'en-tête du classement
    for rang, (nom_support, nb) in enumerate(top10.items(), start=1):  # Parcourt le Top 10 en numérotant à partir de 1
        pct = nb / total * 100  # Calcule la part en pourcentage de ce type de support par rapport au total
        print(f"{rang}. {nom_support} : {nb} ({pct:.1f} %)")  # Affiche le rang, le nom du support, le nombre et le pourcentage à 1 décimale


Jointure effectuée sur : STA_NM_ANFR

Total : 199053

Top 10 :
1. Non renseigné : 99981 (50.2 %)
2. Pylône autostable : 44336 (22.3 %)
3. Bâtiment : 15752 (7.9 %)
4. Immeuble : 14214 (7.1 %)
5. Pylône tubulaire : 9245 (4.6 %)
6. Château d'eau - réservoir : 3887 (2.0 %)
7. Mât : 1816 (0.9 %)
8. Mât métallique : 1429 (0.7 %)
9. Monument religieux : 1092 (0.5 %)
10. Tunnel : 1072 (0.5 %)
